In [ ]:
!pip install langchain langchain-community langchain-google-genai langchain-huggingface sentence-transformers faiss-cpu python-dotenv rapidocr-onnxruntime

'uv' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# LangChain's Gemini integration checks GOOGLE_API_KEY first.
# If your .env uses GEMINI_API_KEY, this maps it automatically.
google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if not google_api_key:
    raise ValueError("Add GOOGLE_API_KEY=your_gemini_api_key or GEMINI_API_KEY=your_gemini_api_key to your .env file")

os.environ["GOOGLE_API_KEY"] = google_api_key

## Data Ingestion


In [3]:
from langchain.document_loaders import TextLoader

ModuleNotFoundError: No module named 'langchain'

In [ ]:
loader = TextLoader("/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

In [ ]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'\ufeffUnderstanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to '

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [ ]:
text_chunks=text_splitter.split_documents(documents)

In [ ]:
text_chunks

[Document(metadata={'source': '/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt'}, page_content='\ufeffUnderstanding Agentic AI'),
 Document(metadata={'source': '/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': '/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'),
 Document(metadata={'source': '/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/Agentic AI.txt'}, page_content='Key Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional 

In [ ]:
! uv pip install faiss-cpu


Using Python 3.12.10 environment at: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/.venv
Resolved 3 packages in 333ms                                         
Installed 1 package in 3ms                                  
 + faiss-cpu==1.12.0


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
# Free local embedding model. No OpenAI key needed.
# Dimension: 384
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [ ]:
vectorstore

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
Key Characteristics of Agentic AI
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
--------------------------------------------------
Document 2:
﻿Understanding Agentic AI
--------------------------------------------------
Document 3:
* Adaptability and Learning: They can adjust their behavior and improve their performance based on new information and experiences.
How Agentic AI Works
--------------------------------------------------
Document 4:
such as ensuring safety, interpretability, and ethical decision-making remain critical areas of research and development for the widespread adoption of agentic AI.
--------------------------------------------------


In [ ]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [ ]:
prompt=ChatPromptTemplate.from_template(template)

In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [ ]:
from langchain.schema.output_parser import StrOutputParser

In [ ]:
output_parser=StrOutputParser()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini LLM. Uses GOOGLE_API_KEY / GEMINI_API_KEY from the .env setup cell.
llm_model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2,
)

In [ ]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [ ]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI represents a new paradigm in artificial intelligence where systems are designed to operate autonomously rather than merely responding to queries or performing specific tasks. These systems exhibit several key characteristics that differentiate them from traditional AI models. One of the main features is their ability to make independent decisions to achieve set goals. The potential applications of agentic AI are extensive and cover various industries, including robotics, where autonomous robots can perform complex tasks in manufacturing, exploration, or logistics. Overall, agentic AI aims to enhance the capabilities of AI systems, allowing them to function more independently and effectively in diverse environments.'

In [ ]:
import structlog

ModuleNotFoundError: No module named 'structlog'